In [3]:
import pandas as pd
import sqlite3

1.) Total de ingresos, costos y utilidad (KPIs Principales) de la tabla de ventas.

In [18]:
# Conexión a la base de datos
conexion = sqlite3.connect("ventas.db")

kpis_query = """
SELECT
    SUM(Quantity * UnitPrice) AS TotalIngresos,
    SUM(Quantity * UnitCost)  AS TotalCostos,
    SUM( (Quantity * UnitPrice) - (Quantity * UnitCost) ) AS TotalUtilidad
FROM
    ventas
"""

kpis_df = pd.read_sql_query(kpis_query, conexion)
conexion.close()

# Formato de moneda
display(
    kpis_df.style.format("${:,.2f}")
)

,TotalIngresos,TotalCostos,TotalUtilidad
0,"$237,645,981.05","$98,590,924.49","$139,055,056.56"


2.) Ingresos totales por año y mes por ventas.

In [19]:
conexion = sqlite3.connect("ventas.db")

ingresos_por_tiempo_query = """
SELECT
    strftime('%Y', OrderDate) AS Ano,
    strftime('%m', OrderDate) AS Mes,
    SUM(Quantity * UnitPrice) AS TotalIngresos
FROM
    ventas
GROUP BY
    Ano, Mes
ORDER BY
    Ano, Mes
"""

ingresos_por_tiempo_df = pd.read_sql_query(ingresos_por_tiempo_query, conexion)
conexion.close()

# Formato
display(
    ingresos_por_tiempo_df.style.format({"TotalIngresos": "${:,.2f}"})
)

,Ano,Mes,TotalIngresos
0,2016,05,"$318,990.83"
1,2016,06,"$748,798.93"
2,2016,07,"$624,390.32"
3,2016,08,"$876,971.17"
4,2016,09,"$682,161.61"
5,2016,10,"$628,018.74"
6,2016,11,"$906,874.81"
7,2016,12,"$1,199,021.06"
8,2017,01,"$867,359.83"
9,2017,02,"$1,132,909.85"


3.) Utilidad por canal de ventas

In [20]:
conexion = sqlite3.connect("ventas.db")

utilidad_por_canal_query = """
SELECT
    CASE
        WHEN StoreName = 'Online store' THEN 'Online'
        ELSE 'Offline'
    END AS CanalDeVentas,
    SUM( (Quantity * UnitPrice) - (Quantity * UnitCost) ) AS Utilidad
FROM
    ventas
GROUP BY
    CanalDeVentas
ORDER BY
    Utilidad DESC
"""

utilidad_por_canal_df = pd.read_sql_query(utilidad_por_canal_query, conexion)
conexion.close()

display(
    utilidad_por_canal_df.style.format({"Utilidad": "${:,.2f}"})
)

,CanalDeVentas,Utilidad
0,Offline,"$83,275,453.08"
1,Online,"$55,779,603.48"


4.) Ventas por región

In [22]:
conexion = sqlite3.connect("ventas.db")

ventas_por_region_query = """
SELECT
    StoreCountryName AS Region,
    SUM(Quantity * UnitPrice) AS TotalVentas
FROM
    ventas
WHERE
    StoreName != 'Online store'
GROUP BY
    Region
ORDER BY
    TotalVentas DESC
"""

ventas_por_region_df = pd.read_sql_query(ventas_por_region_query, conexion)
conexion.close()

display(
    ventas_por_region_df.style.format({"TotalVentas": "${:,.2f}"})
)

,Region,TotalVentas
0,United States,"$72,127,900.58"
1,Canada,"$19,288,423.70"
2,Germany,"$13,892,849.96"
3,United Kingdom,"$12,534,619.38"
4,Australia,"$11,543,398.31"
5,Netherlands,"$5,007,590.57"
6,Italy,"$4,394,099.47"
7,France,"$3,604,932.40"


5.) Top 5 productos con mayor margen de ganacia.

In [23]:
conexion = sqlite3.connect("ventas.db")

top_5_productos_margen_query = """
SELECT
    ProductCode,
    ProductBrand,
    SUM(Quantity * UnitPrice) AS TotalIngresosProducto,
    SUM(Quantity * UnitCost) AS TotalCostosProducto,
    ROUND(
        (SUM(Quantity * UnitPrice) - SUM(Quantity * UnitCost)) * 100.0
        / SUM(Quantity * UnitPrice),
    2) AS MargenGananciaPorcentaje
FROM
    ventas
GROUP BY
    ProductCode, ProductBrand
HAVING
    TotalIngresosProducto > 0
ORDER BY
    MargenGananciaPorcentaje DESC
LIMIT 5
"""

top_5_productos_margen_df = pd.read_sql_query(top_5_productos_margen_query, conexion)
conexion.close()

display(
    top_5_productos_margen_df.style.format({
        "TotalIngresosProducto" : "${:,.2f}",
        "TotalCostosProducto"   : "${:,.2f}",
        "MargenGananciaPorcentaje": "{:.2f}%"
    })
)

,ProductCode,ProductBrand,TotalIngresosProducto,TotalCostosProducto,MargenGananciaPorcentaje
0,602015,Southridge Video,"$30,642.02","$10,147.07",66.89%
1,602020,Southridge Video,"$26,387.62","$8,738.23",66.89%
2,602025,Southridge Video,"$31,953.41","$10,581.34",66.89%
3,602060,Smart-life,"$32,786.71","$10,857.29",66.89%
4,602065,Smart-life,"$30,300.09","$10,033.84",66.89%


6.) Productos por debajo del 20% de margen

In [17]:
conexion = sqlite3.connect("ventas.db")

productos_bajo_margen_query = """
SELECT
    ProductCode,
    ProductBrand,
    SUM(Quantity * UnitPrice)                           AS TotalIngresosProducto,
    SUM(Quantity * UnitCost)                            AS TotalCostosProducto,
    ROUND(
        (SUM(Quantity * UnitPrice) - SUM(Quantity * UnitCost)) * 100.0
        / SUM(Quantity * UnitPrice),
    2)                                                  AS MargenGananciaPorcentaje
FROM
    ventas
GROUP BY
    ProductCode, ProductBrand
HAVING
    MargenGananciaPorcentaje < 20
ORDER BY
    MargenGananciaPorcentaje DESC
"""

productos_bajo_margen_df = pd.read_sql_query(productos_bajo_margen_query, conexion)
conexion.close()

display(productos_bajo_margen_df)

,ProductCode,ProductBrand,TotalIngresosProducto,TotalCostosProducto,MargenGananciaPorcentaje


In [28]:
conexion = sqlite3.connect("ventas.db")

productos_margen_query = """
SELECT
    ProductCode,
    ProductBrand,
    ROUND(
        (SUM(Quantity * UnitPrice) - SUM(Quantity * UnitCost)) * 100.0
        / SUM(Quantity * UnitPrice),
    2) AS MargenGananciaPorcentaje
FROM
    ventas
GROUP BY
    ProductCode, ProductBrand
ORDER BY
    MargenGananciaPorcentaje ASC
"""

productos_margen_df = pd.read_sql_query(productos_margen_query, conexion)
conexion.close()

# Estadísticas del margen
print(productos_margen_df["MargenGananciaPorcentaje"].describe())

count    2517.000000
mean       54.585058
std         6.421090
min        48.950000
25%        49.020000
50%        54.010000
75%        54.020000
max        66.890000
Name: MargenGananciaPorcentaje, dtype: float64


Por eso el HAVING < 20 no devolvía nada — ningún producto está ni cerca de ese umbral.

7.) Crecimiento de los ingresos mensuales del mes actual respecto al anterior para 2024 y 2025

In [24]:
conexion = sqlite3.connect("ventas.db")

crecimiento_ingresos_query = """
SELECT
    strftime('%Y', OrderDate) AS Ano,
    strftime('%m', OrderDate) AS Mes,
    SUM(Quantity * UnitPrice) AS TotalIngresos
FROM
    ventas
WHERE
    strftime('%Y', OrderDate) IN ('2024', '2025')
GROUP BY
    Ano, Mes
ORDER BY
    Ano, Mes
"""

ingresos_2024_2025_df = pd.read_sql_query(crecimiento_ingresos_query, conexion)
conexion.close()

ingresos_2024_2025_df['CrecimientoMensualPorcentaje'] = (
    ingresos_2024_2025_df.groupby('Ano')['TotalIngresos'].pct_change() * 100
)

meses = {
    "01": "Enero", "02": "Febrero", "03": "Marzo",
    "04": "Abril", "05": "Mayo", "06": "Junio",
    "07": "Julio", "08": "Agosto", "09": "Septiembre",
    "10": "Octubre", "11": "Noviembre", "12": "Diciembre"
}
ingresos_2024_2025_df["Mes"] = ingresos_2024_2025_df["Mes"].map(meses)

display(
    ingresos_2024_2025_df.style.format({
        "TotalIngresos"              : "${:,.2f}",
        "CrecimientoMensualPorcentaje": "{:+.2f}%"  # + muestra el signo positivo/negativo
    }).highlight_null(props="color: gray;")          # NaN de enero se ve en gris
)

,Ano,Mes,TotalIngresos,CrecimientoMensualPorcentaje
0,2024,Enero,"$3,619,474.65",+nan%
1,2024,Febrero,"$4,601,965.73",+27.14%
2,2024,Marzo,"$2,265,097.52",-50.78%
3,2024,Abril,"$1,305,966.35",-42.34%
4,2024,Mayo,"$3,372,131.86",+158.21%
5,2024,Junio,"$2,886,871.57",-14.39%
6,2024,Julio,"$2,414,313.53",-16.37%
7,2024,Agosto,"$2,936,673.92",+21.64%
8,2024,Septiembre,"$2,525,047.06",-14.02%
9,2024,Octubre,"$2,869,817.42",+13.65%


8.) Calcula cuánto gasta un cliente en promedio por cada compra.

In [25]:
conexion = sqlite3.connect("ventas.db")

# Consulta para calcular el gasto promedio por compra
# Primero, calculamos el total de cada compra (OrderKey)
# Luego, calculamos el promedio de esos totales
gasto_promedio_compra_query = """
SELECT
    AVG(TotalGastoPorCompra) AS GastoPromedioCompra
FROM
    (
        SELECT
            OrderKey,
            SUM(Quantity * UnitPrice) AS TotalGastoPorCompra
        FROM
            ventas
        GROUP BY
            OrderKey
    ) AS SubconsultaGastoPorCompra
"""

gasto_promedio_compra_df = pd.read_sql_query(gasto_promedio_compra_query, conexion)
conexion.close()

display(
    gasto_promedio_compra_df.style.format({"GastoPromedioCompra": "${:,.2f}"})
)

,GastoPromedioCompra
0,"$2,542.48"


9.) Calcula el porcentaje que representa cada producto sobre el total global.

In [26]:
conexion = sqlite3.connect("ventas.db")

porcentaje_productos_query = """
SELECT
    ProductCode,
    ProductBrand,
    SUM(Quantity * UnitPrice) AS TotalVentasProducto,
    ROUND(
        SUM(Quantity * UnitPrice) * 100.0 / (SELECT SUM(Quantity * UnitPrice) FROM ventas),
    2) AS PorcentajeSobreTotalGlobal
FROM
    ventas
GROUP BY
    ProductCode, ProductBrand
ORDER BY
    PorcentajeSobreTotalGlobal DESC
"""

porcentaje_productos_df = pd.read_sql_query(porcentaje_productos_query, conexion)
conexion.close()

display(
    porcentaje_productos_df.style.format({
        "TotalVentasProducto"      : "${:,.2f}",
        "PorcentajeSobreTotalGlobal": "{:.2f}%"
    })
)

,ProductCode,ProductBrand,TotalVentasProducto,PorcentajeSobreTotalGlobal
0,303001,Adventure Works,"$2,063,441.89",0.87%
1,303023,Wide World Importers,"$2,002,792.21",0.84%
2,303013,Adventure Works,"$1,924,638.81",0.81%
3,303007,Adventure Works,"$1,897,684.73",0.80%
4,303040,Wide World Importers,"$1,870,612.23",0.79%
5,303029,Wide World Importers,"$1,847,978.86",0.78%
6,303018,Adventure Works,"$1,827,111.11",0.77%
7,303035,Wide World Importers,"$1,789,037.54",0.75%
8,201031,Adventure Works,"$1,541,307.39",0.65%
9,303008,Adventure Works,"$1,415,922.89",0.60%


10.) Días de la semana con más ventas en promedio para 2025.

In [27]:
conexion = sqlite3.connect("ventas.db")

ventas_dias_semana_2025_query = """
SELECT
    CASE strftime('%w', OrderDate)
        WHEN '0' THEN 'Domingo'
        WHEN '1' THEN 'Lunes'
        WHEN '2' THEN 'Martes'
        WHEN '3' THEN 'Miércoles'
        WHEN '4' THEN 'Jueves'
        WHEN '5' THEN 'Viernes'
        WHEN '6' THEN 'Sábado'
    END AS DiaDeLaSemana,
    AVG(Quantity * UnitPrice) AS VentasPromedio
FROM
    ventas
WHERE
    strftime('%Y', OrderDate) = '2025'
GROUP BY
    DiaDeLaSemana
ORDER BY
    VentasPromedio DESC
"""

ventas_dias_semana_2025_df = pd.read_sql_query(ventas_dias_semana_2025_query, conexion)
conexion.close()

display(
    ventas_dias_semana_2025_df.style.format({
        "VentasPromedio": "${:,.2f}"
    })
)

,DiaDeLaSemana,VentasPromedio
0,Jueves,$888.67
1,Lunes,$882.69
2,Viernes,$878.14
3,Miércoles,$869.01
4,Martes,$863.43
5,Domingo,$850.16
6,Sábado,$841.87
